# Mini-TP 7 · Seguridad con SAIF — Aprendizaje federado

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este es **uno de los seis notebooks guiados** del Mini-TP 7. Cada uno asegura **una capa**
de las que construimos en el bimestre. **Elige este si tu modelo se expone por Aprendizaje federado**
(o simplemente para practicar esta estrategia). Con completar **uno** alcanza para la entrega.

### SAIF en esta capa
**Riesgo:** **clientes maliciosos** que envenenan el modelo global y **fuga de datos** por los updates.
**Controles:** agregación robusta (mediana/trimmed) · *clipping* de la norma · privacidad diferencial.
**Elementos SAIF:** detectar y actuar · mejora continua.

### Qué vas a hacer
1. Ver la **versión vulnerable** y por qué es un riesgo.
2. Aplicar el **control principal** (guiado, ya resuelto como ejemplo).
3. Completar los **`# TODO`** sobre tu propio modelo.
4. Escribir una **reflexión** (amenaza mitigada ↔ elemento de SAIF).

> Corre de punta a punta con **uv**, sin servicios externos.

### Dependencias
```bash
uv add scikit-learn numpy pandas xgboost
```

## 0. Mi modelo y mis datos
Igual que en los otros Mini-TP, uso **mis datos**: los autos usados de Car Dekho con el
feature engineering de `MLOps1_final/common` (las 23 features de `xgb_best`, target
`log(precio)`).

El modelo federado es una **regresión lineal**, igual que en el Mini-TP 5 y que el
`linreg_federado` que entrena el servicio federado del TP integrador: un XGBoost no se puede
promediar coordenada a coordenada (son árboles), así que FedAvg no aplica. Entreno también
`xgb_best` centralizado, solo como referencia de cuánto se puede lograr con estos datos.

In [1]:
import os, sys, time, json, hashlib, datetime
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

# Reusar el preprocesamiento del TP integrador (MLOps1_final/common)
TP_MLOPS1 = os.path.abspath(os.path.join("..", "TP_Final", "MLOps1_final"))
if TP_MLOPS1 not in sys.path:
    sys.path.insert(0, TP_MLOPS1)
DATASET_CSV = os.path.join(TP_MLOPS1, "dataset", "Car details v3.csv")

from common.constants import XGB_PARAMS, TRAIN_TEST_SIZE, TRAIN_TEST_SPLIT_SEED, MLFLOW_MODEL_NAME
from common.preprocessing import (
    extraer_columnas_tecnicas, drop_missing_technical_rows, drop_km_outliers,
    compute_top_brands, encode_categoricals, get_all_feature_columns,
    align_columns, build_feature_matrix, build_inference_features,
)
from common.schemas import CarRawInput

# Dataset y split estratificado por rango de precio (igual que en el TP integrador)
df = pd.read_csv(DATASET_CSV).drop_duplicates().reset_index(drop=True)
price_bins = pd.qcut(df["selling_price"], q=5,
                     labels=["muy_barato", "barato", "intermedio", "caro", "muy_caro"])
df_train_raw, df_test_raw = train_test_split(df, test_size=TRAIN_TEST_SIZE,
                                             stratify=price_bins, random_state=TRAIN_TEST_SPLIT_SEED)

def preparar(df_tr_raw, df_te_raw):
    """Feature engineering del TP integrador: devuelve X/y de train y test + metadata."""
    df_tr = drop_km_outliers(drop_missing_technical_rows(extraer_columnas_tecnicas(df_tr_raw)))
    df_te = drop_missing_technical_rows(extraer_columnas_tecnicas(df_te_raw))
    brands = compute_top_brands(df_tr)
    tr_enc, te_enc = encode_categoricals(df_tr, df_te, brands)
    feats = get_all_feature_columns(tr_enc)
    tr_enc, te_enc = align_columns(tr_enc, te_enc, feats)
    return (build_feature_matrix(tr_enc, feats), np.log1p(tr_enc["selling_price"]),
            build_feature_matrix(te_enc, feats), te_enc["selling_price"], brands, feats, df_tr, df_te)

X_train, y_train, X_test, y_test, top_brands, all_features, df_train, df_test = preparar(df_train_raw, df_test_raw)

modelo = XGBRegressor(**XGB_PARAMS)
modelo.fit(X_train, y_train)          # el modelo predice log(precio)

pred_test = np.expm1(modelo.predict(X_test))
METRICAS = {"r2": float(r2_score(y_test, pred_test)), "mae": float(mean_absolute_error(y_test, pred_test))}
MODEL_VERSION = f"{MLFLOW_MODEL_NAME}-local"
print(f"Modelo {MLFLOW_MODEL_NAME} listo. R2 test = {METRICAS['r2']:.4f} | MAE test = {METRICAS['mae']:,.0f}"
      f" | n_features = {len(all_features)}")

def predecir_precio(car: CarRawInput) -> float:
    """Mismo camino que la API del TP integrador: auto crudo -> features -> precio."""
    fila = build_inference_features(car, top_brands, all_features)
    return float(np.expm1(modelo.predict(fila)[0]))

COLS_AUTO = ["name", "year", "km_driven", "fuel", "seller_type", "transmission", "owner",
             "mileage", "engine", "max_power", "torque", "seats"]
# Auto de ejemplo (primer auto del dataset, precio real 450.000)
AUTO = {
    "name": "Maruti Swift Dzire VDI", "year": 2014, "km_driven": 145500,
    "fuel": "Diesel", "seller_type": "Individual", "transmission": "Manual",
    "owner": "First Owner", "mileage": "23.4 kmpl", "engine": "1248 CC",
    "max_power": "74 bhp", "torque": "190Nm@ 2000rpm", "seats": 5,
}
print(f"Precio predicho para el auto de ejemplo: {predecir_precio(CarRawInput(**AUTO)):,.0f}")


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

esc = StandardScaler().fit(X_train)
Xtr = esc.transform(X_train); ytr = y_train.to_numpy()
Xte = esc.transform(X_test);  yte = np.log1p(y_test.to_numpy())
N_FEAT = Xtr.shape[1]

def metricas(w, b):
    pred = Xte @ w + b
    rmse = float(np.sqrt(np.mean((pred - yte) ** 2)))
    mae = float(np.mean(np.abs(np.expm1(np.clip(pred, 0, 20)) - np.expm1(yte))))
    return rmse, mae

lin = LinearRegression().fit(Xtr, ytr)
RMSE_CENTRAL, MAE_CENTRAL = metricas(lin.coef_, lin.intercept_)
rmse_xgb = float(np.sqrt(np.mean((np.log1p(pred_test) - yte) ** 2)))
print(f"\nRegresión lineal centralizada: RMSE(log) = {RMSE_CENTRAL:.3f} | MAE = {MAE_CENTRAL:,.0f}")
print(f"xgb_best centralizado (ref.) : RMSE(log) = {rmse_xgb:.3f} | MAE = {METRICAS['mae']:,.0f}")

Modelo xgb_best listo. R2 test = 0.9372 | MAE test = 72,408 | n_features = 23
Precio predicho para el auto de ejemplo: 455,294

Regresión lineal centralizada: RMSE(log) = 0.260 | MAE = 101,933
xgb_best centralizado (ref.) : RMSE(log) = 0.210 | MAE = 72,408


## 1. FedAvg y un cliente MALICIOSO
Reparto el train en **5 concesionarias** (clientes). Cada ronda, cada cliente parte del modelo
global, entrena unas épocas con sus datos y manda su **update** (la diferencia con el global).
El cliente 0 es **malicioso**: manda su update **escalado ×10 e invertido** (*model poisoning*).

La función `federar` ya tiene todas las opciones que voy a usar en los TODO (clipping, privacidad
diferencial, media recortada, exclusión de clientes), para comparar todo con el mismo código.

In [2]:
rng = np.random.RandomState(0)
K = 5
CLIENTES = [(Xtr[p], ytr[p]) for p in np.array_split(rng.permutation(len(Xtr)), K)]
print("Datos por cliente:", [len(c[1]) for c in CLIENTES])

def entrenar_local(theta, Xc, yc, epocas=5, lr=0.1):
    """Regresión lineal con descenso por gradiente. theta = [w..., b]."""
    w, b = theta[:-1].copy(), theta[-1]
    for _ in range(epocas):
        err = Xc @ w + b - yc
        w -= lr * Xc.T @ err / len(yc); b -= lr * err.mean()
    return np.append(w, b)

def recortar(U, S):
    """Clipping: si la norma de un update supera S, se reescala a norma S."""
    normas = np.linalg.norm(U, axis=1, keepdims=True)
    return U * np.minimum(1, S / np.maximum(normas, 1e-12))

def agregar(U, agg, trim=1):
    if agg == "media":
        return U.mean(0)
    if agg == "mediana":
        return np.median(U, 0)
    if agg == "trimmed":                       # media recortada por coordenada
        Us = np.sort(U, 0)
        return Us[trim:len(U) - trim].mean(0)
    raise ValueError(agg)

def federar(agg="media", atacantes=(), R=30, clip=False, sigma=0.0, S_fijo=None, excluir=False, log=None, seed=0):
    """clip: recorta cada update a S. S = S_fijo si se da (necesario para DP), si no la mediana de la ronda."""
    if sigma > 0 and not (clip and S_fijo):
        raise ValueError("la privacidad diferencial necesita clipping con un límite FIJO (S_fijo)")
    rs = np.random.RandomState(seed)
    theta = np.zeros(N_FEAT + 1)
    sospechas = np.zeros(K, int); excluidos = set()
    for r in range(R):
        activos = [i for i in range(K) if i not in excluidos]
        U = []
        for i in activos:
            u = entrenar_local(theta, *CLIENTES[i]) - theta
            if i in atacantes:                       # cliente malicioso
                u = -10 * u
            U.append(u)
        U = np.array(U)
        normas = np.linalg.norm(U, axis=1)
        S = np.median(normas)                        # norma "típica" de la ronda
        # --- detección: norma muy grande o dirección opuesta al consenso (mediana) ---
        ref = np.median(U, 0)
        cos = U @ ref / (normas * np.linalg.norm(ref) + 1e-12)
        for j, i in enumerate(activos):
            sospechoso = normas[j] > 3 * S or cos[j] < 0
            sospechas[i] += sospechoso
            if log is not None:
                log.append({"ronda": r, "cliente": i, "norma": round(float(normas[j]), 4),
                            "cos_consenso": round(float(cos[j]), 3), "sospechoso": bool(sospechoso),
                            "excluido_despues": bool(excluir and sospechas[i] >= 3 and i not in excluidos)})
            if excluir and sospechas[i] >= 3:
                excluidos.add(i)
        S_clip = S_fijo or S
        if clip:
            U = recortar(U, S_clip)
        delta = agregar(U, agg)
        if sigma > 0:                                # privacidad diferencial (gaussiana)
            delta = delta + rs.normal(0, sigma * S_clip / len(U), size=delta.shape)
        theta = theta + delta
    rmse, mae = metricas(theta[:-1], theta[-1])
    return {"rmse": rmse, "mae": mae, "excluidos": sorted(excluidos)}

def fila(nombre, res):
    rmse = res["rmse"]
    estado = "OK" if rmse < 1.2 * RMSE_CENTRAL else ("DEGRADADO" if rmse < 3 * RMSE_CENTRAL else "COLAPSA")
    mae = f"{res['mae']:>12,.0f}" if rmse < 5 else f"{'---':>12}"
    print(f"  {nombre:42s} RMSE(log) = {rmse:10.3f}   MAE = {mae}   {estado}")

fila("sin ataque · FedAvg (media)", federar("media"))
fila("con ataque · FedAvg (media)", federar("media", atacantes={0}))

Datos por cliente: [1007, 1007, 1006, 1006, 1006]
  sin ataque · FedAvg (media)                RMSE(log) =      0.262   MAE =      103,009   OK
  con ataque · FedAvg (media)                RMSE(log) = 1939742616.648   MAE =          ---   COLAPSA


## 2. Control principal (GUIADO) — agregación robusta (mediana por coordenada)

In [3]:
fila("con ataque · robusto (mediana)", federar("mediana", atacantes={0}))
print("\nLa mediana por coordenada ignora los valores extremos del cliente malicioso.")
print("Es una defensa de 'detectar y actuar': tolera clientes que no son de confianza.")

  con ataque · robusto (mediana)             RMSE(log) =      0.265   MAE =      104,172   OK

La mediana por coordenada ignora los valores extremos del cliente malicioso.
Es una defensa de 'detectar y actuar': tolera clientes que no son de confianza.


## 3. Tu entrega — completa los `# TODO`

- **TODO 1 — clipping:** antes de agregar, si la norma del update de un cliente supera `S`, se
  reescala a norma `S`. Uso como `S` la **mediana de las normas de la ronda** (*clipping
  adaptativo*): los updates normales casi no se tocan y el del atacante queda del mismo tamaño
  que los demás.
- **TODO 2 — privacidad diferencial:** ruido gaussiano sobre el update agregado, con desvío
  `sigma · S / K` (el esquema de DP-FedAvg: el clipping acota cuánto puede aportar un cliente, y
  el ruido tapa ese aporte). Acá el límite `S` tiene que ser **fijo** (uso 0.5). Lo probé primero
  con el `S` adaptativo y el modelo **divergía**: el ruido agranda los updates, eso agranda `S`,
  y eso agranda el ruido. Además, para que la garantía de DP valga, `S` no puede depender de los
  datos. La función ahora lo exige.
- **TODO 3 — media recortada:** por coordenada, saco el valor más alto y el más bajo (de 5) y
  promedio el resto. Lo comparo con la mediana, también con **2** atacantes.
- **TODO 4 — mi modelo/dataset:** todo corre sobre mis datos de autos repartidos en 5 clientes.
- **Gobernanza (detectar y actuar):** cada ronda registro, por cliente, la norma de su update y
  su similitud (coseno) con el consenso (la mediana). Si un cliente es sospechoso 3 rondas,
  queda **excluido** de la federación.

In [4]:
S_DP = 0.5        # límite fijo de clipping para DP (los updates honestos rondan 0.6 en régimen)
print("TODO 1 · clipping (adaptativo: S = mediana de las normas de la ronda)")
fila("1 atacante · clipping + media", federar("media", atacantes={0}, clip=True))
fila("1 atacante · clipping + mediana", federar("mediana", atacantes={0}, clip=True))

print("\nTODO 3 · media recortada vs mediana")
for n_at in (1, 2):
    at = set(range(n_at))
    fila(f"{n_at} atacante(s) · media", federar("media", atacantes=at))
    fila(f"{n_at} atacante(s) · mediana", federar("mediana", atacantes=at))
    fila(f"{n_at} atacante(s) · media recortada (1 por lado)", federar("trimmed", atacantes=at))
    fila(f"{n_at} atacante(s) · clipping + media recortada", federar("trimmed", atacantes=at, clip=True))

print(f"\nTODO 2 · privacidad diferencial (clipping fijo S = {S_DP}, promedio de 5 corridas)")
for sigma in (0.0, 0.05, 0.1, 0.25, 0.5):
    sin = [federar("media", clip=True, S_fijo=S_DP, sigma=sigma, seed=s)["rmse"] for s in range(5)]
    con = [federar("mediana", atacantes={0}, clip=True, S_fijo=S_DP, sigma=sigma, seed=s)["rmse"] for s in range(5)]
    print(f"  sigma = {sigma:<5} RMSE(log) sin ataque (media) = {np.mean(sin):.3f} | "
          f"1 atacante (mediana) = {np.mean(con):.3f}")
try:
    federar("media", clip=True, sigma=0.1)
except ValueError as e:
    print("  DP con clipping adaptativo ->", e)

TODO 1 · clipping (adaptativo: S = mediana de las normas de la ronda)
  1 atacante · clipping + media              RMSE(log) =      0.266   MAE =      105,049   OK
  1 atacante · clipping + mediana            RMSE(log) =      0.265   MAE =      103,862   OK

TODO 3 · media recortada vs mediana
  1 atacante(s) · media                      RMSE(log) = 1939742616.648   MAE =          ---   COLAPSA
  1 atacante(s) · mediana                    RMSE(log) =      0.265   MAE =      104,172   OK
  1 atacante(s) · media recortada (1 por lado) RMSE(log) =      0.265   MAE =      104,063   OK
  1 atacante(s) · clipping + media recortada RMSE(log) =      0.266   MAE =      104,357   OK
  2 atacante(s) · media                      RMSE(log) = 3748616331678503424.000   MAE =          ---   COLAPSA
  2 atacante(s) · mediana                    RMSE(log) =      0.266   MAE =      106,294   OK
  2 atacante(s) · media recortada (1 por lado) RMSE(log) = 57328060313184.922   MAE =          ---   COLAPSA
  2

In [5]:
# Gobernanza: registro por ronda y por cliente + exclusión automática
LOG = []
res = federar("media", atacantes={0}, clip=True, excluir=True, log=LOG)
fila("1 atacante · clipping + media + exclusión", res)
print("Clientes excluidos:", res["excluidos"])

log = pd.DataFrame(LOG)
print("\nResumen del registro (30 rondas):")
print(log.groupby("cliente").agg(norma_media=("norma", "mean"), cos_medio=("cos_consenso", "mean"),
                                 rondas_sospechoso=("sospechoso", "sum"), rondas=("ronda", "count")))
log[log["cliente"] == 0].head(4)

  1 atacante · clipping + media + exclusión  RMSE(log) =      0.262   MAE =      103,178   OK
Clientes excluidos: [0]

Resumen del registro (30 rondas):
         norma_media  cos_medio  rondas_sospechoso  rondas
cliente                                                   
0          41.153167  -0.991333                  3       3
1           0.608437   0.727100                  0      30
2           0.601860   0.623333                  0      30
3           0.605503   0.633733                  0      30
4           0.603673   0.613000                  0      30


,ronda,cliente,norma,cos_consenso,sospechoso,excluido_despues
0,0,0,52.9902,-0.992,True,False
5,1,0,40.1021,-0.991,True,False
10,2,0,30.3672,-0.991,True,True


In [6]:
# Configuración final: clipping fijo + mediana + DP + exclusión, con 0, 1 y 2 atacantes
print(f"Configuración final (clipping S={S_DP} + mediana + DP sigma=0.1 + exclusión):")
for n_at in (0, 1, 2):
    r = federar("mediana", atacantes=set(range(n_at)), clip=True, S_fijo=S_DP, sigma=0.1, excluir=True)
    fila(f"{n_at} atacante(s) · excluidos {r['excluidos']}", r)
print(f"\nReferencia centralizada: RMSE(log) = {RMSE_CENTRAL:.3f} | MAE = {MAE_CENTRAL:,.0f}")

Configuración final (clipping S=0.5 + mediana + DP sigma=0.1 + exclusión):
  0 atacante(s) · excluidos []               RMSE(log) =      0.281   MAE =      112,481   OK
  1 atacante(s) · excluidos [0]              RMSE(log) =      0.285   MAE =      115,116   OK
  2 atacante(s) · excluidos [0, 1]           RMSE(log) =      0.288   MAE =      118,754   OK

Referencia centralizada: RMSE(log) = 0.260 | MAE = 101,933


## 4. Reflexión

**1. ¿Por qué el promedio simple es tan frágil ante un solo cliente malicioso?**
Porque en un promedio **cada cliente puede mover el resultado tanto como quiera**: no hay ningún
límite a su peso. Con 5 clientes, 4 honestos mandan `u` y el atacante manda `-10u`; el promedio es
`(4u - 10u) / 5 = -1.2u`: el modelo global **retrocede** en cada ronda y diverge (el RMSE explota).
Un solo valor extremo alcanza para dominar la media. La mediana, en cambio, no se mueve por un
valor extremo: con 5 clientes tolera hasta 2 atacantes. La media recortada (1 por lado) tolera 1,
y con 2 atacantes colapsa (con clipping, en cambio, solo se degrada), como se ve en la tabla.

**2. ¿Qué protege el *clipping* y qué protege la privacidad diferencial (son cosas distintas)?**
- El **clipping** protege el **modelo** (integridad): limita cuánto puede **mover** el modelo un
  solo cliente en una ronda. El atacante ya no puede mandar un update 10 veces más grande; queda
  del tamaño de los demás. Con 1 atacante, el clipping solo ya alcanzó: su update invertido resta
  lo que suma un cliente honesto, así que el modelo aprende más lento, pero aprende. Con 2
  atacantes ya no alcanza (la media recortada con clipping se degradó), y hace falta una
  agregación robusta o la exclusión.
- La **privacidad diferencial** protege los **datos de los clientes** (confidencialidad): el
  ruido hace que el modelo global no dependa demasiado de los datos de ningún cliente en
  particular, y así sus updates no se pueden usar para reconstruir o inferir sus datos (por
  ejemplo, qué autos vendió una concesionaria). Tiene un costo en calidad: más `sigma`, más error. Con solo 5 clientes el costo es alto (el
  ruido por cliente es `sigma · S / K`): `sigma = 0.1` casi no se nota, pero `0.5` ya empeora
  bastante el modelo. Con cientos de clientes, el mismo nivel de privacidad costaría mucho menos.
- Se complementan: el ruido de DP se calibra con `S`, así que **sin clipping no hay DP** (sin
  un límite al aporte de cada cliente, no hay cuánto ruido alcance).

**3. ¿Qué elemento de SAIF cubre mi control principal?**

| Control | Amenaza | Elemento de SAIF |
|---|---|---|
| Agregación robusta (mediana / media recortada) | clientes maliciosos, *model poisoning* | **Detectar y actuar** |
| Clipping de la norma | updates gigantes | **Automatizar defensas** |
| Privacidad diferencial | fuga de datos de los clientes por los updates | **Bases sólidas** (protección de datos) |
| Registro por ronda + exclusión tras 3 sospechas | atacante persistente sin que nadie se entere | **Detectar y actuar** + gobernanza |
| Comparar configuraciones con 0, 1 y 2 atacantes | defensas que se rompen con un atacante más | **Mejora continua** (*adapt controls*) |

El control principal (la mediana) cubre **detectar y actuar**: el sistema asume que algunos
clientes pueden no ser de confianza y **tolera** sus updates sin necesidad de saber quiénes son.
Con el registro por ronda doy el paso siguiente: además de tolerar, **detecto** (norma y dirección
del update) y **actúo** (excluyo al cliente). Y la comparación de configuraciones contra 1 y 2
atacantes es **mejora continua**: muestra que la media recortada que alcanzaba con 1 atacante deja
de alcanzar con 2, y que hay que ajustar los controles al riesgo real.